# 실습 문제 모범답안 - 310. Subagents Pattern

`310_Subagents_Pattern` 노트북 실습 문제(미니 개인 비서 만들기)에 대한 모범답안 예시입니다.

### 구현 요약

본문의 함수를 재사용하지 않고 3계층 구조를 처음부터 직접 구현합니다.

1. 스텁 도구 2개 정의: `get_weather`, `create_memo`
2. 하위 에이전트 2개 생성: `weather_agent`, `memo_agent`
3. `check_weather`, `manage_memo` 도구로 래핑 → 슈퍼바이저 생성 → 복합 요청 테스트

In [6]:
from dotenv import load_dotenv

load_dotenv()

True

In [7]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-mini", model_provider="openai")

## 1. 스텁 도구 2개 정의

실제로는 날씨 API(기상청, OpenWeather 등)와 메모 앱 API를 호출하지만,
여기서는 정해진 문자열만 반환하는 스텁(stub)으로 구현합니다.

In [8]:
from langchain.tools import tool

@tool
def get_weather(
    city: str,  # 도시 이름
    date: str   # ISO 형식 날짜: "2024-01-15"
) -> str:
    """지정한 도시와 날짜의 날씨를 조회합니다."""
    # 실제로는 기상청, OpenWeather 등의 날씨 API를 호출합니다.
    return f"{city} {date} 날씨: 맑음, 기온 5도"

@tool
def create_memo(
    title: str,   # 메모 제목
    content: str  # 메모 내용
) -> str:
    """새로운 메모를 저장합니다."""
    # 실제로는 Notion, Keep 등의 메모 앱 API를 호출합니다.
    return f"메모 저장 완료: {title}"

## 2. 하위 에이전트 2개 생성

각 도구를 사용하는 도메인별 전문 에이전트를 만듭니다.

In [9]:
from langchain.agents import create_agent

weather_agent = create_agent(
    model,
    tools=[get_weather],
    system_prompt=(
        "당신은 날씨 안내 도우미입니다. "
        "자연어 요청에서 도시와 날짜를 추출하세요. "
        "날짜는 정확한 ISO 형식(예: 2024-01-15)으로 변환하고, "
        "get_weather 도구로 날씨를 조회해 알려주세요."
    ),
)

memo_agent = create_agent(
    model,
    tools=[create_memo],
    system_prompt=(
        "당신은 메모 작성 도우미입니다. "
        "자연어 요청에서 적절한 메모 제목과 내용을 구성하고, "
        "create_memo 도구로 메모를 저장하세요. "
        "마지막 응답에서 어떤 메모가 저장되었는지 확인시켜 주세요."
    ),
)

## 3. 래핑 + 슈퍼바이저 + 복합 요청 테스트

하위 에이전트를 도구로 래핑하고, 두 도구를 조율하는 슈퍼바이저를 만듭니다.
복합 요청을 주면 슈퍼바이저가 `check_weather`와 `manage_memo`를 각각 호출하는지 확인하세요.

In [10]:
@tool
def check_weather(request: str) -> str:
    """자연어 요청을 기반으로 날씨를 조회합니다. 예: '내일 서울 날씨 알려줘'"""
    result = weather_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].content

@tool
def manage_memo(request: str) -> str:
    """자연어 요청을 기반으로 메모를 작성/저장합니다. 예: '장보기 목록 메모해줘'"""
    result = memo_agent.invoke({
        "messages": [{"role": "user", "content": request}]
    })
    return result["messages"][-1].content


mini_assistant = create_agent(
    model,
    tools=[check_weather, manage_memo],
    system_prompt=(
        "당신은 친절한 개인 비서입니다. "
        "날씨를 조회하고 메모를 저장할 수 있습니다. "
        "사용자의 요청을 적절한 도구 호출로 분해하고, 그 결과를 조율하세요."
    ),
)

# 복합 요청 테스트 (날씨 조회 + 메모 저장)
query = "내일 서울 날씨를 확인하고, 우산을 챙기라는 메모를 남겨줘"

print("=" * 80)
print("미니 개인 비서 테스트 (날씨 + 메모):")
print("=" * 80)

for step in mini_assistant.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

미니 개인 비서 테스트 (날씨 + 메모):
================================ Human Message =================================

내일 서울 날씨를 확인하고, 우산을 챙기라는 메모를 남겨줘
================================== Ai Message ==================================
Tool Calls:
  check_weather (call_P9lt2yCXPLxwNG5BYPavzfK5)
 Call ID: call_P9lt2yCXPLxwNG5BYPavzfK5
  Args:
    request: 내일 서울 날씨 알려줘
  manage_memo (call_aCSVeJWWhuu5VRpwB8m5s0tt)
 Call ID: call_aCSVeJWWhuu5VRpwB8m5s0tt
  Args:
    request: 내일 우산 챙기라고 메모해줘
================================= Tool Message =================================
Name: manage_memo

메모를 저장했습니다.

저장된 메모:
- 제목: 내일 우산 챙기기 (2026-07-06)
- 내용: 2026-07-06: 외출 시 우산을 꼭 챙기기. 가방이나 현관에 두어 잊지 않기.
================================== Ai Message ==================================

완료했습니다.

- 날씨(내일, 2026-07-06) — 서울: 맑음, 기온 5°C. (추가로 강수확률·습도·시간대별 예보 원하시면 알려주세요.)
- 메모 저장됨:
  - 제목: 내일 우산 챙기기 (2026-07-06)
  - 내용: 2026-07-06: 외출 시 우산을 꼭 챙기기. 가방이나 현관에 두어 잊지 않기.

다음에 알림 설정(시간에 맞춰 푸시) 해드릴까요? 또는 날씨 상세 정보를 더 볼까요?
